In [ ]:
!pip install -q datasets scikit-learn joblib

## Import libraries

In [49]:
import time
import os
import joblib
import re
import numpy as np
import pandas as pd

from datasets import load_dataset, concatenate_datasets
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import train_test_split

## Donwload dataset from Hugging Face

In [38]:
ds = load_dataset("contemmcm/sentiment140")

ds

DatasetDict({
    complete: Dataset({
        features: ['text', 'label'],
        num_rows: 1600000
    })
})

Here we rename the columns

In [39]:
data = ds["complete"]

TEXT_COL = "text"
LABEL_COL = "label"

We define a seed for repoducibility and work with a subsample from the original datasets to save resources.

In [40]:
SUBSAMPLE_SIZE = 50_000
SEED = 42

Here we filter positive and negative labels and shuffle the sub dataset.

In [41]:
negative = data.filter(lambda x: x["label"] == 0)
positive = data.filter(lambda x: x["label"] == 1)

n_per_class = SUBSAMPLE_SIZE // 2

negative_sample = negative.shuffle(seed=SEED).select(range(n_per_class))
positive_sample = positive.shuffle(seed=SEED).select(range(n_per_class))

balanced_data = concatenate_datasets([negative_sample, positive_sample]).shuffle(seed=SEED)

print("Balanced sample size:", len(balanced_data))

Balanced sample size: 50000


We store everything in a DataFrame to make visualization and manipulation easier.

In [42]:
df = pd.DataFrame({
    "text": balanced_data["text"],
    "label": [0 if y == 0 else 1 for y in balanced_data["label"]]
})

df.head()

,text,label
0,I'm actually pretty sad the Duke has left us ...,0
1,"@Heatherrrr_ haha spazzz ;) and i know, i've o...",0
2,@christina1986 what cam did you get now? the ...,1
3,ha! one line css grids framework. http://bit....,1
4,"GOODMORNING... Ugh, twitter fully kicked me of...",0


In [43]:
X = df["text"]
y = df["label"]

We add a simple function to clean tweets by removing hashtags and noisy text

In [44]:
def clean_tweet(text):
    text = str(text)

    # Strip URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Replace @usernames with @user
    text = re.sub(r"@\w+", "@user", text)

    # Normalize extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

df["clean_text"] = df["text"].apply(clean_tweet)

df[["text", "clean_text", "label"]].head(10)

,text,clean_text,label
0,I'm actually pretty sad the Duke has left us ...,I'm actually pretty sad the Duke has left us W...,0
1,"@Heatherrrr_ haha spazzz ;) and i know, i've o...","@user haha spazzz ;) and i know, i've only jus...",0
2,@christina1986 what cam did you get now? the ...,@user what cam did you get now? the one that i...,1
3,ha! one line css grids framework. http://bit....,ha! one line css grids framework. (via @user) ...,1
4,"GOODMORNING... Ugh, twitter fully kicked me of...","GOODMORNING... Ugh, twitter fully kicked me of...",0
5,Itchy eyes,Itchy eyes,0
6,off to school tom. which is soo not good. i ha...,off to school tom. which is soo not good. i ha...,0
7,"@cdncamel LOL, Claire. Even if you pass on th...","@user LOL, Claire. Even if you pass on the roo...",1
8,Really feel like crap this morning feel like ...,Really feel like crap this morning feel like I...,0
9,@peterloggins hehe i know! Luka is the BEST fr...,@user hehe i know! Luka is the BEST friend ever!,1


We split the dataset into training and testing partitions to facilitate model training.

In [45]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Train label balance:")
print(y_train.value_counts())
print("Test label balance:")
print(y_test.value_counts())

Train size: 40000
Test size: 10000
Train label balance:
label
1    20000
0    20000
Name: count, dtype: int64
Test label balance:
label
0    5000
1    5000
Name: count, dtype: int64


In [46]:
df = pd.DataFrame({
    "text": balanced_data["text"],
    "label": [0 if y == 0 else 1 for y in balanced_data["label"]]
})

df.head()

,text,label
0,I'm actually pretty sad the Duke has left us ...,0
1,"@Heatherrrr_ haha spazzz ;) and i know, i've o...",0
2,@christina1986 what cam did you get now? the ...,1
3,ha! one line css grids framework. http://bit....,1
4,"GOODMORNING... Ugh, twitter fully kicked me of...",0


We use word-level and character-level TF-IDF vectorizers with n-grams, combined with a logistic regression classifier, as a classic machine learning approach for sentiment classification.

In [50]:
classic_model = Pipeline([
    ("features", FeatureUnion([
        ("word_tfidf", TfidfVectorizer(
            lowercase=True,
            analyzer="word",
            ngram_range=(1, 2),      # word unigrams + bigrams
            min_df=2,
            max_features=80_000
        )),
        ("char_tfidf", TfidfVectorizer(
            lowercase=True,
            analyzer="char",
            ngram_range=(2, 5),      # character bigrams to 5-grams
            min_df=2,
            max_features=80_000
        ))
    ])),
    ("clf", LogisticRegression(
        max_iter=1000,
        solver="liblinear",
        random_state=SEED
    ))
])

classic_model

Pipeline(steps=[('features',
                 FeatureUnion(transformer_list=[('word_tfidf',
                                                 TfidfVectorizer(max_features=80000,
                                                                 min_df=2,
                                                                 ngram_range=(1,
                                                                              2))),
                                                ('char_tfidf',
                                                 TfidfVectorizer(analyzer='char',
                                                                 max_features=80000,
                                                                 min_df=2,
                                                                 ngram_range=(2,
                                                                              5)))])),
                ('clf',
                 LogisticRegression(max_iter=1000, random_state=42,
                                    solver='liblinear'))])

We fit the model

In [51]:
start_time = time.time()

classic_model.fit(X_train, y_train)

train_time = time.time() - start_time

print(f"Training time: {train_time:.2f} seconds")

Training time: 22.44 seconds


Then we run the predictions

In [52]:
y_pred = classic_model.predict(X_test)

We measure the typical classification evaluations metrics including Accuracy and F1 score.

In [53]:
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 score: {f1:.4f}")

Accuracy: 0.7968
F1 score: 0.7983


We add a deeper report using the classification_report function from sklearn library.

In [55]:
print(classification_report(
    y_test,
    y_pred,
    target_names=["negative", "positive"]
))

              precision    recall  f1-score   support

    negative       0.80      0.79      0.80      5000
    positive       0.79      0.80      0.80      5000

    accuracy                           0.80     10000
   macro avg       0.80      0.80      0.80     10000
weighted avg       0.80      0.80      0.80     10000



Also, we review the confusion matrix to obserse false positives and false negative in the performance of our classifier.

In [56]:
cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Negative", "Actual Positive"],
    columns=["Predicted Negative", "Predicted Positive"]
)

cm_df

,Predicted Negative,Predicted Positive
Actual Negative,3946,1054
Actual Positive,978,4022


We measure the inference time employ in our model.

In [57]:
sample_texts = X_test.tolist()

start_time = time.time()

_ = classic_model.predict(sample_texts)

total_inference_time = time.time() - start_time
latency_per_example_ms = (total_inference_time / len(sample_texts)) * 1000

print(f"Total inference time: {total_inference_time:.4f} seconds")
print(f"Inference latency: {latency_per_example_ms:.4f} ms/example")

Total inference time: 2.5341 seconds
Inference latency: 0.2534 ms/example


We evaluate the model size in terms of the memory it occupies on the machine

In [58]:
model_path = "classic_tfidf_logreg_sentiment140.joblib"

joblib.dump(classic_model, model_path)

model_size_mb = os.path.getsize(model_path) / (1024 * 1024)

print("Model saved to:", model_path)
print(f"Model size: {model_size_mb:.2f} MB")

Model saved to: classic_tfidf_logreg_sentiment140.joblib
Model size: 5.64 MB


We summarize all the results in a table for our classic machine learning classifier.

In [59]:
classic_results = {
    "Model": "TF-IDF + Logistic Regression",
    "Dataset Size": len(df),
    "Train Size": len(X_train),
    "Test Size": len(X_test),
    "Accuracy": accuracy,
    "F1": f1,
    "Train Time (s)": train_time,
    "Inference (ms/example)": latency_per_example_ms,
    "Model Size (MB)": model_size_mb
}

classic_results_df = pd.DataFrame([classic_results])

classic_results_df

,Model,Dataset Size,Train Size,Test Size,Accuracy,F1,Train Time (s),Inference (ms/example),Model Size (MB)
0,TF-IDF + Logistic Regression,50000,40000,10000,0.7968,0.798333,22.441771,0.253406,5.638535


We store all the results in csv file to compare with other approaches.

In [60]:
classic_results_df.to_csv("classic_model_results.csv", index=False)

print("Saved results to classic_model_results.csv")

Saved results to classic_model_results.csv
